In [ ]:
# Colab bootstrap: install requirements and stage the course data next to the notebook
import shutil
import subprocess
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    REPO_URL = "https://github.com/griu/ai-ml-finance-hands-on"
    REPO_DIR = Path("/content/ai-ml-finance-hands-on")
    NOTEBOOK_DIR = Path.cwd()
    DATA_TARGET = NOTEBOOK_DIR / "data"
    REQUIREMENTS_FILE = REPO_DIR / "requirements-colab.txt"

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_FILE)],
        check=True,
    )

    shutil.copytree(REPO_DIR / "data", DATA_TARGET, dirs_exist_ok=True)
    print(f"Colab environment detected. Data copied to: {DATA_TARGET.resolve()}")
else:
    print("Local environment detected. Colab bootstrap skipped.")


# 02 — Python Warmup for Finance Students

**Course:** Data Analysis, AI and Machine Learning in Finance  
**Session:** 1 (main practical — approx. 1 h 30 min)  
**Instructor:** Ferran Carrascosa  
**Dataset:** `dataset_A_python_warmup.csv` (50 rows × 8 columns)

---

## Why this notebook matters

Python is a practical language for reading and transforming tables—exactly the kind of work
you will do throughout this course:

- **Exploratory Data Analysis (EDA)** — understanding what the data looks like.
- **Statistical modelling** — testing hypotheses, building baselines.
- **Machine learning** — from logistic regression to XGBoost and beyond.

This notebook is **not** a programming course. It is a guided warm-up that takes you from
"I can run a cell" to "I can manipulate simple data and understand what the code is doing."

> **Goal:** build confidence, not depth.

## How to read this notebook

Notebook 02 is not a programming exam. It is a guided transition from **basic Python objects** to **data structures used in finance analytics**.

The aim is to make students think in this repeated pattern:

- **Diagnose** — what kind of object am I working with?
- **Transform** — how do I create or modify it?
- **Validate** — how do I inspect what happened?
- **Justify** — why is this useful in a real finance workflow?

Keep that pattern in mind throughout the notebook. It is more important than memorising every line of syntax.

---

## Section 1 — Basic Python Refresher

A quick tour of the data types you will use most often.

In [1]:
# Numbers
revenue = 250_000          # integer
growth_rate = 0.075        # float (7.5 %)
print("Revenue:", revenue)
print("Expected next-year revenue:", revenue * (1 + growth_rate))

Revenue: 250000
Expected next-year revenue: 268750.0


In [2]:
# Strings (text)
bank_name = "Meridian Bank"
print(f"Client of {bank_name}")

Client of Meridian Bank


In [3]:
# Booleans (True / False)
is_profitable = revenue > 200_000
print("Is the company profitable?", is_profitable)

Is the company profitable? True


In [4]:
# Lists — ordered collections
products = ["Savings", "Loan", "Investment", "Current"]
print("Products offered:", products)
print("First product:", products[0])
print("Number of products:", len(products))

Products offered: ['Savings', 'Loan', 'Investment', 'Current']
First product: Savings
Number of products: 4


In [5]:
# Dictionaries — key-value pairs
customer = {
    "name": "Ana Torres",
    "segment": "Premium",
    "balance": 45_000,
}
print(customer["name"], "—", customer["segment"], "— balance:", customer["balance"])

Ana Torres — Premium — balance: 45000


---

## Section 2 — Simple Control Flow

We rarely write raw loops when working with pandas, but understanding `if/else` and basic
iteration helps when creating flags or business rules.

In [6]:
# if / else — classify a balance
balance = 32_000

if balance >= 30_000:
    risk_label = "low risk"
elif balance >= 10_000:
    risk_label = "medium risk"
else:
    risk_label = "high risk"

print(f"Balance {balance:,} EUR → {risk_label}")

Balance 32,000 EUR → low risk


In [7]:
# A short loop — iterate over a list of returns
quarterly_returns = [0.03, -0.01, 0.05, 0.02]

for i, r in enumerate(quarterly_returns, start=1):
    print(f"  Q{i}: {r:+.1%}")

  Q1: +3.0%
  Q2: -1.0%
  Q3: +5.0%
  Q4: +2.0%


In [8]:
# Combining logic — assign risk labels to several customers
balances = [5_000, 15_000, 42_000, 8_000, 30_000]

for b in balances:
    label = "low" if b >= 30_000 else ("medium" if b >= 10_000 else "high")
    print(f"  {b:>6,} EUR → {label}")

   5,000 EUR → high
  15,000 EUR → medium
  42,000 EUR → low
   8,000 EUR → high
  30,000 EUR → low


> **Note:** in the next sections we will do the same kind of logic *inside* a DataFrame,
> without writing explicit loops. pandas handles the repetition for us.

### From Python objects to analytical tables

The first sections of the notebook look simple on purpose. Lists, dictionaries and conditional logic are the building blocks behind later analytical work.

A dataframe can be understood as a **programmable business table**. Before students can manipulate one confidently, they need to understand the smaller structures from which analytical logic is built.

---

## Section 3 — Loading the Dataset with pandas

Time to leave toy examples behind and work with a real (small) dataset.

**Dataset A** contains 50 fictitious retail-banking customers with information about their
balance, transaction volume, fees, product type, and channel preference.

In [9]:
import pandas as pd
from pathlib import Path

# Resolve data directory — works from the project root or notebooks/
_candidates = [Path("data/processed"), Path("../data/processed")]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Cannot locate data/processed/. "
        "Launch the notebook from the project root or the notebooks/ folder."
    )

df = pd.read_csv(DATA_DIR / "dataset_A_python_warmup.csv")

print("Dataset loaded successfully.")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")


Dataset loaded successfully.
Shape: 50 rows × 8 columns


In [ ]:
# First five rows
df.head()

In [ ]:
# Column names
print("Columns:", list(df.columns))

In [ ]:
# Data types
df.dtypes

**What do we see?**

| Column | Meaning |
|--------|---------|
| `customer_id` | Unique customer identifier |
| `age_group` | Age bracket (e.g., 18-25, 26-35, …) |
| `region` | Geographic region (North, South, East, West) |
| `product_type` | Main banking product (Savings, Loan, Investment, Current) |
| `monthly_balance` | Average monthly balance (EUR) |
| `monthly_transactions` | Number of transactions per month |
| `fee_paid` | Monthly fee paid (EUR) |
| `digital_channel_flag` | 1 = primarily digital, 0 = primarily branch |

> **Takeaway:** "This is my table and I know what kind of information it contains."

### Diagnose before transforming

At this point the important question is not yet “what model should we use?” The important question is:

**what kind of table are we looking at, and what can it already tell us?**

In practical finance work, this first inspection step avoids many later mistakes. It tells us whether the columns make sense, whether the field names are readable and whether the table already has the shape required for analysis.

---

## Section 4 — Selecting and Filtering Data

In [ ]:
# Select a single column
df["monthly_balance"].head(10)

In [ ]:
# Select several columns at once
df[["customer_id", "product_type", "monthly_balance"]].head()

In [ ]:
# Filter: customers with a balance above 30 000 EUR
high_balance = df[df["monthly_balance"] > 30_000]
print(f"Customers with balance > 30 000: {len(high_balance)} out of {len(df)}")
high_balance[["customer_id", "product_type", "monthly_balance"]]

In [ ]:
# Combine filters: digital customers in the South region
south_digital = df[(df["region"] == "South") & (df["digital_channel_flag"] == 1)]
print(f"Digital customers in the South: {len(south_digital)}")
south_digital.head()

In [ ]:
# Sort by monthly balance (descending)
df.sort_values("monthly_balance", ascending=False).head(10)

---

## Section 5 — Creating New Columns

One of the most common tasks in data analysis: turning raw data into **business-meaningful
variables**.

In [ ]:
# Fee-to-balance ratio — how much fee relative to balance
df["fee_ratio"] = df["fee_paid"] / df["monthly_balance"]
df[["customer_id", "fee_paid", "monthly_balance", "fee_ratio"]].head()

In [ ]:
# High-value customer flag — balance above the median
median_balance = df["monthly_balance"].median()
df["high_value"] = (df["monthly_balance"] >= median_balance).astype(int)

print(f"Median balance: {median_balance:,.0f} EUR")
print()
print(df["high_value"].value_counts())

In [ ]:
# Transaction intensity category
df["txn_category"] = pd.cut(
    df["monthly_transactions"],
    bins=[0, 15, 40, 100],
    labels=["Low", "Medium", "High"],
)
df["txn_category"].value_counts()

> We now have three new columns: `fee_ratio`, `high_value`, and `txn_category`.  
> Each one answers a clear business question.

### Why derived columns matter

Creating a new column is not a cosmetic coding step. It is often the moment when raw fields start becoming **analytical variables**.

For example:
- a ratio may express intensity better than a raw amount,
- a flag may simplify a business rule,
- a grouped category may make later analysis easier to communicate.

A good derived variable should therefore always answer a question:
**what analytical meaning did we create that was not visible before?**

---

## Section 6 — Descriptive Summaries

Before any modelling, we must **understand** the data.

In [ ]:
# Overall numeric summary
df.describe()

In [ ]:
# Category counts
print("=== Product type ===")
print(df["product_type"].value_counts())
print()
print("=== Age group ===")
print(df["age_group"].value_counts())

In [ ]:
# Missing values check
df.isna().sum()

In [ ]:
# Grouped summary — mean balance by product type
df.groupby("product_type")["monthly_balance"].mean().sort_values(ascending=False)

In [ ]:
# Grouped summary — mean transactions by age group
df.groupby("age_group")["monthly_transactions"].agg(["mean", "median", "count"])

---

## Section 7 — First Plots

A chart often reveals patterns that summary numbers alone cannot show.

In [ ]:
import matplotlib.pyplot as plt

# Histogram — distribution of monthly balance
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["monthly_balance"], bins=10, color="steelblue", edgecolor="white")
ax.set_xlabel("Monthly Balance (EUR)")
ax.set_ylabel("Number of Customers")
ax.set_title("Distribution of Monthly Balance")
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart — average balance by product type
avg_by_product = df.groupby("product_type")["monthly_balance"].mean().sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
avg_by_product.plot.barh(ax=ax, color="darkorange", edgecolor="white")
ax.set_xlabel("Average Monthly Balance (EUR)")
ax.set_title("Average Balance by Product Type")
plt.tight_layout()
plt.show()

**Interpretation prompts:**

- Which product type has the highest average balance?
- Is the balance distribution roughly symmetric, or is it skewed?
- Are there very few customers in any group?

### Validate and justify the first plots

At this stage, the plots are intentionally simple. Their purpose is not visual sophistication. Their purpose is to make students practise a key analytical habit:

1. **look at the output**,
2. **state what pattern is visible**,
3. **check whether it is plausible**,
4. **explain why the pattern matters for a finance workflow**.

This habit will become more important in Notebook 03, where descriptive statistics and plots turn into a first real EDA workflow.

---

## Section 8 — Guided Business Reading

Answer these questions by looking at the outputs above (and running additional cells if needed).
Write your answers in a markdown cell or on paper.

1. **Which product type has the most customers?** Is the dataset balanced across products?
2. **Which region has the highest average monthly balance?** Could this be explained by
   the number of customers in that region?
3. **Do digital-channel customers (`digital_channel_flag == 1`) pay higher fees on average?**
   What might explain that pattern?
4. **Which age group has the highest median number of transactions?**
   Does this match your business intuition?
5. **Looking at the histogram of monthly balance,** would you say the distribution is roughly
   symmetric or skewed? What does that mean for typical versus extreme customers?
6. **If you had to predict which customers generate higher fees,** which two or three
   variables would you start with? Why?

> These are the kinds of questions you will learn to answer systematically in the coming sessions.

---

## Section 9 — Mini Consolidation Task

Now it is your turn. In the cells below, try to reproduce a small analytical workflow on
your own:

1. **Load** the dataset again (you may copy the `read_csv` line from Section 3).
2. **Inspect** the first rows and the shape.
3. **Compute** two grouped summaries of your choice (e.g., mean fee by region, median
   balance by age group).
4. **Create** one new column based on business logic (e.g., a flag, a ratio, a category).
5. **Plot** one chart (histogram or bar chart) and write one sentence of interpretation.

Use the empty cells below as your workspace.

In [ ]:
# --- Your code here ---

In [ ]:
# --- Your code here ---

In [ ]:
# --- Your code here ---

In [ ]:
# --- Your code here ---

---

### Self-practice tasks (do these during the week)

1. Reload the dataset on your own and re-run your analysis from scratch.
2. Create **two additional** derived columns (e.g., a spending intensity index, a region–product combination).
3. Produce **one extra grouped summary** that was not shown in class.
4. Create **one extra plot** (try a box plot: `df.boxplot(column="fee_paid", by="product_type")`).
5. Write **three business interpretations** in English in a markdown cell.

---

*End of Notebook 02 — Python Warmup for Finance Students*

## Bridge to Notebook 03

Notebook 02 shows that Python is not an abstract programming language here. It is the medium through which business tables become analytical objects.

Notebook 03 now uses that same logic on a slightly richer dataset. The student will move from:
- object handling,
- filtering,
- grouped summaries,
- and first plots

to a more explicit **EDA-as-diagnosis** mindset.

That is the moment where Topic 1 starts connecting directly with Topic 2.